## Final Threat Detection Dataset Creation

In [ ]:
import os
import shutil

# OpenImages threat paths (from FiftyOne for images, labels_temp for labels)
openimages_base_dir = "/Users/lvntkymn14/fiftyone/open-images-v7/"
openimages_labels_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/openimages-threat/labels_temp/"

# COCO-threat remapped paths
coco_labels_train_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco-threat/labels_remapped/train/"
coco_labels_val_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco-threat/labels_remapped/val/"
coco_images_train_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco-threat/images/train/"
coco_images_val_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco-threat/images/val/"

# Final merged dataset output path
final_dataset_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-threat-detection-dataset/"

# Subfolders
subfolders = ["images/train", "images/val", "images/test", "labels/train", "labels/val", "labels/test"]

# Create subfolders
for subfolder in subfolders:
    os.makedirs(os.path.join(final_dataset_dir, subfolder), exist_ok=True)

# === Helper Functions ===

def copy_openimages_split(split_name, final_split):
    """
    Copies OpenImages images and labels for a given split ('train', 'validation', 'test') into final split.
    """
    split_dir = os.path.join(openimages_base_dir, split_name, "data")
    label_split = {"train": "train", "validation": "val", "test": "test"}[split_name]
    
    for filename in os.listdir(split_dir):
        if filename.endswith(".jpg"):
            basename = os.path.splitext(filename)[0]
            label_filename = f"{basename}.txt"
            
            # Copy images
            shutil.copy(
                os.path.join(split_dir, filename),
                os.path.join(final_dataset_dir, f"images/{final_split}/", filename)
            )
            
            # Copy labels if exists
            label_path = os.path.join(openimages_labels_dir, label_filename)
            if os.path.exists(label_path):
                shutil.copy(
                    label_path,
                    os.path.join(final_dataset_dir, f"labels/{final_split}/", label_filename)
                )

def copy_coco_split(images_dir, labels_dir, final_split):
    """
    Copies COCO-threat images and labels into final dataset for a given split.
    """
    for filename in os.listdir(images_dir):
        if filename.endswith(".jpg"):
            basename = os.path.splitext(filename)[0]
            label_filename = f"{basename}.txt"
            
            # Copy images
            shutil.copy(
                os.path.join(images_dir, filename),
                os.path.join(final_dataset_dir, f"images/{final_split}/", filename)
            )
            
            # Copy labels
            label_path = os.path.join(labels_dir, label_filename)
            if os.path.exists(label_path):
                shutil.copy(
                    label_path,
                    os.path.join(final_dataset_dir, f"labels/{final_split}/", label_filename)
                )

# Copy OpenImages and COCO-threat splits
print("Copying OpenImages train split...")
copy_openimages_split("train", "train")

print("Copying OpenImages validation split...")
copy_openimages_split("validation", "val")

print("Copying OpenImages test split...")
copy_openimages_split("test", "test")

print("Copying COCO-threat train split...")
copy_coco_split(coco_images_train_dir, coco_labels_train_dir, "train")

print("Copying COCO-threat val split...")
copy_coco_split(coco_images_val_dir, coco_labels_val_dir, "val")

print("Merging complete! Final dataset is at:", final_dataset_dir)

# data.yaml creation
data_yaml_path = os.path.join(final_dataset_dir, "data.yaml")
with open(data_yaml_path, "w") as f:
    f.write(f"train: {os.path.join(final_dataset_dir, 'images/train')}\n")
    f.write(f"val: {os.path.join(final_dataset_dir, 'images/val')}\n")
    f.write(f"test: {os.path.join(final_dataset_dir, 'images/test')}\n\n")
    f.write("nc: 14\n")
    f.write("names: [\"Truck\", \"Baseball bat\", \"Rifle\", \"Weapon\", \"Knife\", \"Missile\", \"Sword\", \"Shotgun\", \"Handgun\", \"Backpack\", \"Dagger\", \"Bow and arrow\", \"Suitcase\", \"Scissors\"]\n")

print("data.yaml created at:", data_yaml_path)

🚀 Copying OpenImages train split...
🚀 Copying OpenImages validation split...
🚀 Copying OpenImages test split...
🚀 Copying COCO-threat train split...
🚀 Copying COCO-threat val split...
✅ Merging complete! Final dataset is at: /Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-threat-detection-dataset/
✅ data.yaml created at: /Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-threat-detection-dataset/data.yaml


In [4]:
import os

def clean_images_without_labels(images_dir, labels_dir):
    images = {os.path.splitext(f)[0] for f in os.listdir(images_dir) if f.endswith(".jpg")}
    labels = {os.path.splitext(f)[0] for f in os.listdir(labels_dir) if f.endswith(".txt")}
    
    unmatched = images - labels
    for img_name in unmatched:
        img_path = os.path.join(images_dir, img_name + ".jpg")
        if os.path.exists(img_path):
            os.remove(img_path)
            print(f"Deleted unmatched image: {img_path}")

# Paths
final_dataset_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-threat-detection-dataset/"

# Run cleanup on train/val/test
splits = ["train", "val", "test"]
for split in splits:
    images_dir = os.path.join(final_dataset_dir, f"images/{split}")
    labels_dir = os.path.join(final_dataset_dir, f"labels/{split}")
    clean_images_without_labels(images_dir, labels_dir)

print("Cleanup complete! All images now have matching labels.")

Cleanup complete! All images now have matching labels.


In [1]:
import os
from collections import Counter

# Paths
final_dataset_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/final-threat-detection-dataset/"
splits = ["train", "val", "test"]

# Final master class names
class_names = [
    "Truck", "Baseball bat", "Rifle", "Weapon", "Knife", "Missile", "Sword",
    "Shotgun", "Handgun", "Backpack", "Dagger", "Bow and arrow", "Suitcase", "Scissors"
]

# Initialize counters
split_counters = {split: Counter() for split in splits}

# Collect counts for each split
for split in splits:
    labels_dir = os.path.join(final_dataset_dir, f"labels/{split}")
    for filename in os.listdir(labels_dir):
        if not filename.endswith(".txt"):
            continue
        filepath = os.path.join(labels_dir, filename)
        with open(filepath, "r") as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            class_id = int(float(parts[0]))
            split_counters[split][class_id] += 1

# Now print table-style output
print("\nCombined Class Counts per Split (Train, Val, Test):\n")
print(f"{'Class Name':<20} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 50)
for class_id, class_name in enumerate(class_names):
    train_count = split_counters["train"].get(class_id, 0)
    val_count = split_counters["val"].get(class_id, 0)
    test_count = split_counters["test"].get(class_id, 0)
    print(f"{class_name:<20} {train_count:>8} {val_count:>8} {test_count:>8}")


Combined Class Counts per Split (Train, Val, Test):

Class Name              Train      Val     Test
--------------------------------------------------
Truck                   22108      769     1072
Baseball bat             4504      168       57
Rifle                    2540      135      404
Weapon                   2960      333     1036
Knife                    8620      406      233
Missile                   603       31       83
Sword                     567       27      123
Shotgun                   580       54      169
Handgun                   727       24       84
Backpack                 9936      403       94
Dagger                    370       50      160
Bow and arrow             594       29       71
Suitcase                 6822      336       86
Scissors                 1880       37        6
